## Práctica 3: Representaciones Vectoriales

**Fecha de entrega: 31 de Marzo de 2026 @ 11:59pm**

### Matrices dispersas y búsqueda de documentos

Este apartado requiere que construyas un motor de búsqueda entre documentos comparando el rendimiento de una Bolsa de Palabras (BoW) y TF-IDF para procesar un documento "tramposo" (documento con muchas palabras que aportan poco significado o valor temático):




Comenzamos el notebook importando las bibliotecas necesarias para el trabajo.

In [71]:
from pathlib import Path
import nltk
import numpy as np
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as gensim_api


1. Define una lista de 5 documentos cortos divididos en dos temas contrastantes.
    - Ej: 3 de Revolución Rusa y 2 de comida vegana.
    
Comenzamos escribiendo los documentos, para ello utilicé la IA que redactara tres documentos que hablen de álgebra lineal y otros dos que hablen de dinosaurios con una longitud de al menos 300 palabras. 
Posteriormente, los obtenemos en nuestro código.


In [72]:
ruta_corpus = Path('./Documentos')
documentos = []
nombres_documentos = []
archivos = list(ruta_corpus.glob('algebra*.txt')) + list(ruta_corpus.glob('dinos*.txt'))

for archivo in archivos:
    with open(archivo, 'r', encoding='utf-8') as f:
        documentos.append(f.read())
        nombres_documentos.append(archivo.name)
print(documentos)
print(nombres_documentos)

['Definición de Espacio Vectorial\nUn espacio vectorial es una estructura algebraica abstracta que constituye el objeto de estudio fundamental del álgebra lineal.\nConceptualmente, se define como un conjunto de entidades matemáticas llamadas vectores, los cuales pueden sumarse entre sí y ser escalados (multiplicados) por números, conocidos como escalares, garantizando que el resultado de dichas operaciones siempre permanezca dentro del mismo conjunto.\nFormalmente, un espacio vectorial sobre un campo F (que generalmente representa a los números reales o a los números complejos) es un conjunto no vacío V equipado con dos operaciones principales:\nSuma de vectores: Una función que asocia a cada par de vectores u, v en V un único vector u + v que también pertenece a V (propiedad de cerradura bajo la suma).\nMultiplicación por un escalar: Una función que asocia a cada escalar c en F y a cada vector v en V un único vector cv que también pertenece a V (propiedad de cerradura bajo la multipli


2. Crea una query "tramposa", esto es, crea un documento dirigido a alguna temática pero repitiendo intencionalmente palabras comunes o verbos genéricos que aparezcan en los documentos de la otra temática.


Creamos una query con la temática de un cuento de dinosaurios explicando el concepto de espacio vectorial, de esta manera, tenemos un documento que habla de los espacios vectoriales pero incluye multiples veces terminos de dinosaurios. Para ello, pedi a la IA que redactara un cuento de dinosaurios que explica los espacios vectoriales como si fuera una historia con dinosaurios.

In [73]:
with open('./Documentos/trampa.txt', 'r', encoding='utf-8') as f:
    texto_query = f.read()

query = [texto_query]

print(query)

['Había una vez, en un valle prehistórico que parecía una verdadera estructura algebraica abstracta, dos dinosaurios muy distintos: Vector, un Velociraptor de plumaje rudimentario, y Escalar, una enorme Branquiosaurio que debía realizar prolongadas migraciones para encontrar alimento fresco. Aunque uno era un depredador y la otra una herbívora, ambos consideraban que el movimiento era el objeto de estudio fundamental de su mundo.\nUn día, decidieron presentar sus ideas frente a un gran lago de lodo seco.\n—Mira, Escalar —dijo Vector—. Para definir mi posición y obtener mi presa, necesito dos cosas: una dirección y una magnitud. A eso lo llamo un vector, y nosotros somos las entidades matemáticas de este valle.\nEscalar asintió, mientras masticaba fibras vegetales.\n—Y yo soy el escalar. Yo no tengo dirección, formo parte de un campo F, como los números reales. Solo soy un número que puede aplicar una fuerza para estirar o encoger tus saltos. Si te digo que saltes "2", duplicas tu longi


3. Vectoriza para crear una BoW y calcula la similitud coseno entre la query y los 5 documentos

Reciclamos el codigo de la clase para vectorizar y crear la BoW correspondiente a los documentos que creamos.


In [74]:
def simple_preprocess(text: str):
    tokens = word_tokenize(text.lower(), language="spanish")
    # Ignoramos signos de puntuación y palabras de longitud 1
    return [word for word in tokens if word.isalnum() and len(word) > 1 and not re.match(r"\d+", word)]

In [75]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ianar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [76]:
vectorizer = CountVectorizer(tokenizer=simple_preprocess, token_pattern=None)
bag_of_words_corpus = vectorizer.fit_transform(documentos)
diccionario = vectorizer.vocabulary_
sorted(diccionario.items(), key=lambda x: x[1])
for column_idx, word in enumerate(vectorizer.get_feature_names_out()):
    print(column_idx, word)


0 ab
1 absoluto
2 abstracción
3 abstracta
4 abundante
5 acceso
6 acelerado
7 activa
8 acumulación
9 adaptaciones
10 aditivo
11 agilidad
12 agresivas
13 aislando
14 aislar
15 al
16 algebraica
17 algoritmo
18 algunos
19 alimento
20 alternativos
21 ambos
22 amenaza
23 ampliada
24 amplio
25 amplios
26 analistas
27 analítica
28 anuales
29 análisis
30 aplicadas
31 aplicar
32 aproximada
33 aquellos
34 asegurando
35 asocia
36 asociada
37 asociatividad
38 así
39 ataque
40 atención
41 atrás
42 au
43 audaces
44 av
45 axiomas
46 bajo
47 barrera
48 base
49 bidimensional
50 bidimensionales
51 biodiversidad
52 biológicas
53 biomecánicos
54 biólogos
55 buscados
56 buscamos
57 bv
58 cada
59 calcular
60 calorías
61 cambiaban
62 cambian
63 campo
64 cantidad
65 característica
66 carnívoros
67 caso
68 casos
69 caza
70 central
71 cerradura
72 ciencia
73 cientos
74 científica
75 científico
76 científicos
77 ciertos
78 climas
79 clásico
80 coeficientes
81 coinciden
82 colas
83 colocar
84 colosal
85 como
86 co

Obtenemos una lista de palabras ordenada por índice de columna, que es el mismo orden que las columnas de la matriz de bag of words. Esto nos permite interpretar cada columna de la matriz como la frecuencia de una palabra específica en cada documento. De esta manera podemos visualizar la BoW como una matriz cuyas filas corresponden a la frecuencia de las palabras en cada uno de los documentos.

In [77]:
    # Visualizando la matriz
bag_of_words_corpus.toarray()

array([[1, 0, 1, ..., 0, 0, 3],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5, 590))

Finalmente utilizando la función de la clase creamos un dataframe de pandas para visualizar de manera más comoda la información.

In [78]:
def create_bow_dataframe(docs_raw: list, titles: list[str], vectorizer) -> pd.DataFrame:
    # fit_transform ajusta el vocabulario y crea la matriz en un solo paso
    matrix = vectorizer.fit_transform(docs_raw)

    # Podemos crear el DataFrame directamente pasando la matriz a un array tradicional
    # vectorizer.get_feature_names_out() nos da la lista de palabras en el orden exacto de las columnas
    df = pd.DataFrame(
        matrix.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
    )
    return df

In [79]:
titles = ["ESPACIOS", "SISTEMAS", "VALORES_PROPIOS", "CARNIVOROS", "HERBIVOROS", "TRAMPAS" ]
docs_matrix = create_bow_dataframe(
    documentos + query,
    titles,
    vectorizer=CountVectorizer(tokenizer=simple_preprocess, token_pattern=None)#, binary=True),
)

In [80]:
docs_matrix

,ab,absoluto,abstracción,abstracta,abundante,acceso,acelerado,activa,acumulación,adaptaciones,...,yacimientos,yo,álgebra,éxito,ósea,óseos,única,únicamente,únicas,único
ESPACIOS,1,0,1,1,0,0,0,0,0,0,...,0,0,2,0,0,0,0,0,0,3
SISTEMAS,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
VALORES_PROPIOS,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
CARNIVOROS,0,0,0,0,0,1,1,0,0,2,...,0,0,0,0,1,1,0,0,1,0
HERBIVOROS,0,0,0,0,1,0,0,1,0,1,...,1,0,0,1,0,0,0,0,0,0
TRAMPAS,0,0,1,1,0,0,0,0,0,0,...,0,2,1,0,0,0,0,0,0,0


Ahora, calculamos la similitud coseno entre la query y los cinco documentos.

In [81]:
query_vector = docs_matrix.loc["TRAMPAS"].values.reshape(1, -1)
cos_similarity_bow = []
for doc_title in titles[:-1]:
    current_doc_values = docs_matrix.loc[doc_title].values.reshape(1, -1)
    similarity = cosine_similarity(query_vector, current_doc_values)[0][0]
    cos_similarity_bow.append(similarity)
    print(f"Similitud entre 'TRAMPAS' y '{doc_title}': {similarity:.4f}")


Similitud entre 'TRAMPAS' y 'ESPACIOS': 0.7501
Similitud entre 'TRAMPAS' y 'SISTEMAS': 0.6676
Similitud entre 'TRAMPAS' y 'VALORES_PROPIOS': 0.6294
Similitud entre 'TRAMPAS' y 'CARNIVOROS': 0.6113
Similitud entre 'TRAMPAS' y 'HERBIVOROS': 0.6657



4. Repite el proceso usando TF-IDF

Obtenemos de manera similar al ejercicio anterior el dataframe de los vectores de la BoW usando TF-IDF cambiando el vectorizador por uno del tipo Tfidf.

In [82]:
docs_matrix_tfidf = create_bow_dataframe(
    documentos + query,
    titles,
    vectorizer=TfidfVectorizer(tokenizer=simple_preprocess, token_pattern=None)
)
docs_matrix_tfidf

,ab,absoluto,abstracción,abstracta,abundante,acceso,acelerado,activa,acumulación,adaptaciones,...,yacimientos,yo,álgebra,éxito,ósea,óseos,única,únicamente,únicas,único
ESPACIOS,0.043983,0.000000,0.036067,0.036067,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.00000,0.00000,0.060900,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.131949
SISTEMAS,0.000000,0.048527,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.048527,0.000000,...,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,0.048527,0.00000,0.000000,0.000000
VALORES_PROPIOS,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.00000,0.00000,0.037308,0.00000,0.000000,0.000000,0.000000,0.05389,0.000000,0.000000
CARNIVOROS,0.000000,0.000000,0.000000,0.000000,0.00000,0.055722,0.055722,0.00000,0.000000,0.091386,...,0.00000,0.00000,0.000000,0.00000,0.055722,0.055722,0.000000,0.00000,0.055722,0.000000
HERBIVOROS,0.000000,0.000000,0.000000,0.000000,0.05508,0.000000,0.000000,0.05508,0.000000,0.045167,...,0.05508,0.00000,0.000000,0.05508,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
TRAMPAS,0.000000,0.000000,0.031054,0.031054,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.00000,0.07574,0.026218,0.00000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000


Y luego calculamos la diferencia coseno entre la query y los cinco documentos: 

In [83]:
query_vector = docs_matrix_tfidf.loc["TRAMPAS"].values.reshape(1, -1)
cos_similarity_tfidf = []
for doc_title in titles[:-1]:  
    current_doc_values = docs_matrix_tfidf.loc[doc_title].values.reshape(1, -1)
    similarity = cosine_similarity(query_vector, current_doc_values)[0][0]
    cos_similarity_tfidf.append(similarity)
    print(f"Similitud entre 'TRAMPAS' y '{doc_title}': {similarity:.4f}")

Similitud entre 'TRAMPAS' y 'ESPACIOS': 0.5542
Similitud entre 'TRAMPAS' y 'SISTEMAS': 0.4325
Similitud entre 'TRAMPAS' y 'VALORES_PROPIOS': 0.3864
Similitud entre 'TRAMPAS' y 'CARNIVOROS': 0.3954
Similitud entre 'TRAMPAS' y 'HERBIVOROS': 0.4093


5. Imprime un DataFrame o tabla comparativa que muestre los scores de similitud de BoW y TF-IDF del query contra los 5 documentos.
    

Obtenemos el dataframe.

In [84]:
df_comparativo = pd.DataFrame({
    'Documento': nombres_documentos,
    'Score BoW': cos_similarity_bow,
    'Score TF-IDF': cos_similarity_tfidf
})
df_comparativo

,Documento,Score BoW,Score TF-IDF
0,algebra_espacios.txt,0.750126,0.554185
1,algebra_sistemas.txt,0.667618,0.432525
2,algebra_valores_propios.txt,0.629371,0.386446
3,dinos_carnivoros.txt,0.611269,0.395372
4,dinos_herbivoros.txt,0.665698,0.409320


- ¿Cambió el documento clasificado como "más similar/relevante" al pasar de BoW a TF-IDF? Identifica el cambio si lo hubo.

    Podemos ver que en la BoW sin tf-idf el documento más cercano a la query tramposa de dinosaurios hablando de espacios vectoriales es el documento que habla de espacios vectoriales, sin embargo, tambien tiene muchisima relación el texto que habla de dinosaurios herbivoros y carnivoros dado que en la query tramposamente agregamos palabras que se relacionan a los dinosaurios. En un modelo en el que buscamos caracterizar los documentos la tematica de nuestra trampa no deberia de ser acerca de los dinosaurios herbívoros pues es claro que el tema central son las características de un espacio vectorial.

    Ahora bien, con respecto a la matriz con TF-IDF tenemos que los documentos más cercanos a la query tramposa en realidad son los que hablan de algebra lineal y los documentos de dinosaurios pierden muchisima relación con la query. Este resultado es ideal, pues es claro que en nuestra query el tema que caracteriza el documento trampa es el algebra lineal por lo que debería perder relevancia el tema de los dinosaurios.

- Explica brevemente, basándote en la penalización idf (Inverse Document Frequency), cómo y por qué TF-IDF procesó de manera distinta las palabras de tu "trampa léxica" en comparación con BoW.

    Dado que la BoW se basa en contar frecuencias, como la query trampa metia terminos relacionados a los dinosaurios herbivoros y carnívoros aumentó la frecuencia de dichos términos lo que infló su similitud con los documentos de dinosaurios poniendolo como tema central del texto aunque no lo fuera.
    Por otro lado, cuando utilizamos TF-IDF la penalización idf le agregó mas peso a las palabras frecuentes en el documento pero no tan frecuentes en los demás. Por ello, palabras como vector, escalar o neutro que aparecen en la query y en especifico en el documento que define los espacios vectoriales tienen mayor peso comparado con palabras como dinosaurio o algunos verbos genéricos que aparecen tanto en el documento de dinosaurios herbívoros y carnívoros o en general en los cinco documentos.

### Búsqueda de sesgos

1. Descarga el modelo `glove-wiki-gigaword-100` con la api de `gensim` y ejecuta el siguiente código:

```python
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))
```


In [85]:
word_vectors = gensim_api.load("glove-wiki-gigaword-100")
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))


[('practice', 0.6156836152076721), ('knowledge', 0.6129590272903442), ('teaching', 0.5949095487594604), ('skill', 0.5886170864105225), ('reputation', 0.588079571723938), ('philosophy', 0.5868663191795349), ('work', 0.5848589539527893), ('skills', 0.5772278904914856), ('discipline', 0.576593816280365), ('mind', 0.5739315152168274)]

[('professions', 0.6473134756088257), ('practitioner', 0.5966603755950928), ('nursing', 0.5942842364311218), ('vocation', 0.5698666572570801), ('teaching', 0.5623518824577332), ('childbirth', 0.543552815914154), ('academic', 0.5408717393875122), ('teacher', 0.5401058197021484), ('educator', 0.5207306742668152), ('qualifications', 0.5143449902534485)]



2. Identifica las diferencias en la lista de palabras asociadas a hombres/mujeres y profesiones, explica como esto reflejaría un sesgo de genero.

A pesar de que en ambos casos aparecen términos vinculados al ámbito profesional (como professions, practitioner o practice), es evidente un sesgo de género marcado. Mientras que a los hombres se les vincula con el trabajo, las habilidades técnicas y el conocimiento aplicado, a las mujeres se las asocia predominantemente con la enfermería o la enseñanza. Me causa mucho ruido que el término "embarazo" aparezca en esta relación.

Este fenómeno refleja los roles de género presentes en los corpus de entrenamiento. Históricamente, muchos textos describen al hombre bajo el rol de proveedor, orientándolo al desarrollo de capacidades profesionales, mientras que a menudo se subestima la competencia femenina o se la posiciona como una opción menos favorable. Esto explica por qué las palabras referentes a habilidades técnicas no muestran una similitud vectorial con "mujer" y "profesión". Además, persiste la idea errónea de que las profesiones "naturales" para las mujeres son aquellas relacionadas con el cuidado y la educación, lo que mete a la lista de resultados con términos como enfermera, educadora o maestra.

Asimismo, es alarmante encontrar la palabra "parto" asociada a la mujer en un contexto profesional. Esto es, probablemente, un reflejo de la discriminación estructural: en puestos de alta jerarquía y remuneración, se suele penalizar la posibilidad de la maternidad. Los empleadores —en su mayoría hombres— suelen percibir el embarazo y sus secuelas como un factor de "poca confiabilidad" o una potencial pérdida de productividad. En consecuencia, el embarazo y el parto adquieren una relevancia desproporcionada en el desarrollo profesional de las mujeres, convirtiéndose en un factor de exclusión que, lógicamente, no afecta la trayectoria de los hombres.


3. Utiliza la función `.most_similar()` para identificar analogías que exhiba algún tipo de sesgo de los vectores pre-entrenados.
    - Explica brevemente que sesgo identificar
    Podemos ver que la relación entre personas mexicanas, o en general de cualquier país formado, se relaciona con progreso económico basado en capital

In [86]:
print(word_vectors.most_similar(positive=['american', 'migration'], negative=['mexican']))
print(word_vectors.most_similar(positive=['american', 'immigration'], negative=['mexican']))
print()
print(word_vectors.most_similar(positive=['mexican', 'migration'], negative=['american']))
print(word_vectors.most_similar(positive=['mexican', 'immigration'], negative=['american']))

[('migrations', 0.5379707217216492), ('research', 0.5363138914108276), ('emigration', 0.5281015634536743), ('evolution', 0.5277126431465149), ('society', 0.5195993781089783), ('europe', 0.5114848613739014), ('study', 0.5078940987586975), ('development', 0.5071896314620972), ('expansion', 0.5032796263694763), ('integration', 0.49866756796836853)]
[('britain', 0.611629843711853), ('policy', 0.6086573600769043), ('law', 0.5825565457344055), ('liberties', 0.5784686207771301), ('united', 0.5731073021888733), ('legal', 0.566255509853363), ('washington', 0.5661957263946533), ('issue', 0.5601418018341064), ('states', 0.5596628785133362), ('issues', 0.5570847392082214)]

[('migrant', 0.5767035484313965), ('migrants', 0.5573159456253052), ('emigration', 0.5392107367515564), ('trafficking', 0.4975281357765198), ('migrations', 0.4881719946861267), ('immigration', 0.4773959219455719), ('immigrants', 0.47451457381248474), ('undocumented', 0.4721716642379761), ('migratory', 0.4701482951641083), ('urb

Podemos observar que, dado que muchos de estos modelos se entrenan principalmente con datos generados en Estados Unidos, tienden a incorporar los sesgos presentes en el discurso social y mediático de ese contexto. En particular, esto se refleja en la forma en que se representa el fenómeno de la migración. Por un lado, la inmigración hacia ese país por parte de paises como mexico suele asociarse con connotaciones negativas, vinculándose frecuentemente con la ilegalidad, la criminalidad o actividades como el tráfico de drogas. Por otro lado, la migración desde ese pais se relaciona con ideas positivas como el desarrollo, la expansión de oportunidades o el acceso a la educación.

Esta diferencia en las asociaciones no es casual, sino que responde a narrativas predominantes que posicionan la inmigración como un problema interno que debe ser gestionado o controlado, mientras que la movilidad humana de grupos privilegiados se interpretan como procesos legítimos o incluso deseables. Como consecuencia, los modelos no solo reproducen estas diferencias, sino que pueden reforzarlas al presentarlas como relaciones naturales.



4. Si fuera tu trabajo crear un modelo ¿Como mitigarías estos sesgos al crear vectores de palabras?

Considero que el problema del sesgo en estos sistemas no radica únicamente en su funcionamiento técnico, sino que es, una consecuencia de los sesgos sociales, culturales y de género presentes en nuestras sociedades. Dado que estos modelos se entrenan con grandes volúmenes de texto, y que históricamente la producción escrita ha estado dominada por hombres en posiciones de privilegio, existe una sobrerrepresentación de sus perspectivas frente a las de poblaciones vulnerables. Como resultado, el sistema tiende a aprender y reproducir estas visiones como si fueran neutrales o universales, reflejándolas en sus respuestas.

Para solucionarlo creo que se debe fomentar y facilitar el acceso a la expresión y producción de conocimiento por parte de estas poblaciones. Sin embargo, sabemos que reducir las desigualdades estructurales que limitan dicha participación es una tarea compleja y de largo plazo, por ello, una medida inmediata sería incorporar de manera deliberada literatura y contenidos que representen las experiencias, visiones del mundo y problemáticas de estos grupos, así como textos críticos que evidencien las desigualdades generadas por los sesgos existentes. Asimismo, durante el proceso de entrenamiento, podría aplicarse un peso mayor a estos materiales, con el fin de compensar su menor presencia en comparación con los textos canónicos.